In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

project_root = Path.cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import GPSICModel, KPCR, rbf_kernel
from src.utils import confusion_matrix, create_connections

np.random.seed(10)

In this tutorial, we will simulate a time-series system, and run the $GP_{SIC}$ method on the simulated system to identify the underlying causal structure. We will also run the novel KPCR model on the simulated data. 

The first step will be to generate the simulated system. We will use the 3-variable Mediator system from the paper for $n=250$. The system is defined as

$$
\begin{align}
q_1(t)= \text{sin}(q_2(t-1)) + 0.001\varepsilon_1(t),\\
q_2(t)= \text{cos}(q_3(t-1)) + 0.01\varepsilon_2(t),\\
q_3(t)= 0.5q_3(t-1) + 0.1\varepsilon_3(t)
\end{align}
$$

where each $\varepsilon_i \sim \mathcal{N}(0, 1)$. The ground truth causal connections are thus $q_3 \rightarrow q_2$, $q_2 \rightarrow q_1$.

In [2]:
def mediator(N):
    q1, q2, q3 = np.zeros(N), np.zeros(N), np.zeros(N)
    W1, W2, W3 = np.random.normal(0, 1, N), np.random.normal(0, 1, N), np.random.normal(0, 1, N)
    for n in range(N-1):
        q1[n+1] = np.sin(q2[n]) + 0.001*W1[n]
        q2[n+1] = np.cos(q3[n]) + 0.01*W2[n]
        q3[n+1] = 0.5*q3[n] + 0.1*W3[n]
    return q1, q2, q3

num_samples = 250 
num_samples_dropout = num_samples + 50 
mc_runs = 5 # Generate MC realizations of the simulated system

i = 0
mc_data = []
while i < mc_runs:    
    q1, q2, q3 = mediator(num_samples_dropout)
    q1 = q1[50:]
    q2 = q2[50:]
    q3 = q3[50:]
    X = np.vstack((q1, q2, q3))
    mc_data.append(X)
    i+=1
data = np.stack(mc_data)
nvar = data.shape[1]
lags = 1 # Ground-truth lag

# Ground-truth connections, zero-indexed
actual_cnnx = [(1,0), (2,1)]
cnnx_list = create_connections(nvar)
actual_neg_cnnx = [tup for tup in cnnx_list if tup not in actual_cnnx]

Then we will run the $GP_{SIC}$ model on each Monte Carlo realization of the simulated system, and evaluate several performance metrics by calling ``confusion_matrix``. The ``GPSICModel`` takes the input data as the shape [number of observations, number of time series] and returns the graph of causal connections [Row: Cause, Column: Effect]. This can be flattened and fed into ``confusion_matrix`` if ground-truth connections are known to evaluate several metrics. 

In [3]:
gp_all_results = []
for i in range(mc_runs):
    gp_data = data[i].T 
    gp_model = GPSICModel(gp_data, num_samples, lags, contemporaneous=False) 
    gc_matrix = gp_model.SIC("GPSIC")
    gp_all_results.append(gc_matrix.flatten())
gp_df = pd.DataFrame(np.array(gp_all_results).astype(int), columns=cnnx_list)
gp_df = confusion_matrix(gp_df, actual_cnnx, actual_neg_cnnx, cnnx_list)
gp_df

,"(0, 0)","(0, 1)","(0, 2)","(1, 0)","(1, 1)","(1, 2)","(2, 0)","(2, 1)","(2, 2)",True Positive,False Positive,True Negative,False Negative,Specificity,Precision,Recall,SHD,F1,Accuracy,Balanced Accuracy
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
mean,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,<NA>,<NA>,<NA>,<NA>,1.0,1.0,1.0,0.0,1.0,1.0,1.0
median,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,<NA>,<NA>,<NA>,<NA>,1.0,1.0,1.0,0.0,1.0,1.0,1.0
variance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0
stddev,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0


For real-data, we may want to run cross-validation on the lag order to determine the optimal lag order. This can be done efficiently using the log pseudo-likelihood, as described in Rasmussen Ch. 5.2.4, such that the GP model need only be fit once per response varaible, instead of $n$ times for hard LOO. This can be done by calling ``optimize_lag_loo`` after instantiating the ``GPSICModel`` class. The parameter ``n_lags`` determines how many lag orders to test from $\{1, \dots, \text{n\_lags}+1\}$. This method will reset the class variables to the optimum lag, reformatting the data accordingly. 

In [4]:
optimal_lag = gp_model.optimize_lag_loo(n_lags=3, one_se=True, contemporaneous=False)
print("Optimal lag: ", optimal_lag)

Optimal lag:  1


Next, we will run the KPCR method on the Mediator simulated system. There are several parameters to note for this method: 
  - ``kernel_func``: kernel type; RBF kernel was used in the simulated experiments
  - ``test``: F-test or Correlation-based index as in KGC method
  - ``alpha``: Significance level for f-test
  - ``eps_filter``: Small constant for filtering small eigenvalues
  - ``Nystrom``: Select whether to use the Nystrom approximation (for many observations) or to use the standard kernelised method
  - ``num_inducing``: Number of inducing points for the Nystrom approximation
  - ``ell_coeff``: constant C for lengthscale heuristic, $\ell$ = C$n_t$m
 

In [5]:
all_decisions = []
for i in range(mc_runs):
    kpcr_data = data[i].T
    Pvalues, Decisions = KPCR(kpcr_data, 
                               rbf_kernel, 
                               n_lags=lags, 
                               test="f-test", 
                               alpha=0.05, 
                               eps_filter=1e-6, 
                               Nystrom=False, 
                               num_inducing=30, 
                               ell_coeff=2)
    dec = np.array(np.round(Decisions,4)).flatten()
    all_decisions.append(dec.astype(int))
unified_df = pd.DataFrame(all_decisions, columns=cnnx_list)
unified_df = confusion_matrix(unified_df, actual_cnnx, actual_neg_cnnx, cnnx_list)
unified_df

,"(0, 0)","(0, 1)","(0, 2)","(1, 0)","(1, 1)","(1, 2)","(2, 0)","(2, 1)","(2, 2)",True Positive,False Positive,True Negative,False Negative,Specificity,Precision,Recall,SHD,F1,Accuracy,Balanced Accuracy
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2,0,7,0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
mean,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,<NA>,<NA>,<NA>,<NA>,1.0,1.0,1.0,0.0,1.0,1.0,1.0
median,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,<NA>,<NA>,<NA>,<NA>,1.0,1.0,1.0,0.0,1.0,1.0,1.0
variance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0
stddev,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0
